# Participants Report

Generates APA-style sentences for demographics, volume, and group order.
Used in the Methods section of both papers.


In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root()))
from core.participants import get_participants

# ==============================================================================
# CONFIGURATION
# ==============================================================================

DATE_SOURCE = "20261016"  # update to match your participants_info CSV date

if sys.platform == "win32":
    root_active  = Path("L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior")
    root_passive = Path("L:/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/outputs/measures/behavior")
else:
    root_active  = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/measures/behavior")
    root_passive = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/measures/behavior")

df_active  = pd.read_csv(root_active  / f"{DATE_SOURCE}_active_participants_info.csv")
df_passive = pd.read_csv(root_passive / f"{DATE_SOURCE}_passive_participants_info.csv")

print(f"Active  : {df_active['participant'].nunique()} participants")
print(f"Passive : {df_passive['participant'].nunique()} participants")
print(f"Columns : {df_active.columns.tolist()}")


Active  : 78 participants
Passive : 61 participants
Columns : ['participant', 'group', 'sex', 'age', 'volume_pct', 'volume_db', 'lvl_highflow']


In [2]:
# ==============================================================================
# Helper
# ==============================================================================

def get_df(paper, paradigm):
    """Return df filtered to included participants for a given paper/paradigm."""
    included = list(get_participants(paper, paradigm).keys())
    df_src   = df_active if paradigm.startswith("active") else df_passive
    return df_src[df_src["participant"].isin(included)].copy()


def demographics_sentence(paper, paradigm):
    df  = get_df(paper, paradigm)
    n   = len(df)

    # Sex
    sex_counts = df["sex"].value_counts()
    n_female   = (df["sex"]    == "F").sum()

    # Age
    age_mean = round(df["age"].mean(), 1)
    age_sd   = round(df["age"].std(),  1)
    age_min  = int(df["age"].min())
    age_max  = int(df["age"].max())

    return (
        f"[{paper.upper()} - {paradigm.upper()}]\n"
        f"{n} participants were included in the analyses "
        f"({n_female} female; "
        f"M = {age_mean} years old, SD = {age_sd}, range {age_min}-{age_max})."
    )


def volume_sentence(paper, paradigm):
    df  = get_df(paper, paradigm)
    vol = df["volume_db"].dropna()
    n        = len(df)
    n_volume = len(vol)
    missing  = [p for p in df["participant"] if df[df["participant"] == p]["volume_db"].isna().all()]

    vmin  = round(vol.min(), 1)
    vmax  = round(vol.max(), 1)
    vmean = round(vol.mean(), 1)
    vstd  = round(vol.std(), 1)

    note = f" (n={n_volume} with volume data; missing: {missing})" if missing else ""
    return (
        f"Sound volume was individually calibrated{note} and ranged from "
        f"{vmin} to {vmax} dB SPL (M = {vmean}, SD = {vstd})."
    )

def game_lvl_sentence(paper, paradigm):
    df = get_df(paper, paradigm)
    game_lvl = df["lvl_highflow"].dropna().astype(int)
    missing = df.loc[df["lvl_highflow"].isna(), "participant"].tolist()

    counts = game_lvl.value_counts().sort_index()
    count_str = "; ".join(f"level {lvl}: n = {n}" for lvl, n in counts.items())

    note = f" (n = {len(game_lvl)} with level data; missing: {', '.join(missing)})" if missing else ""
    return (
        f"Game level was individually calibrated{note}, ranging from "
        f"{game_lvl.min()} to {game_lvl.max()} ({count_str})."
    )


In [3]:
# ==============================================================================
# Group order — flow only
# Groups G1-G8 counterbalancing:
#   G1,G2,G3,G4 = Rankin first  | G5,G6,G7,G8 = Compressed first
#   G1,G2,G5,G6 = Oddball A first | G3,G4,G7,G8 = Oddball B first
#   G1,G3,G5,G7 = Low flow first  | G2,G4,G6,G8 = High flow first
# ==============================================================================

def group_order_sentence(paradigm):
    df = get_df("flow", paradigm)
    n  = len(df)

    # Parse group number
    df["group_num"] = df["group"].str.extract(r'(\d+)').astype(int)

    rankin_first     = df["group_num"].isin([1,2,3,4]).sum()
    compressed_first = df["group_num"].isin([5,6,7,8]).sum()

    oddball_a_first  = df["group_num"].isin([1,2,5,6]).sum()
    oddball_b_first  = df["group_num"].isin([3,4,7,8]).sum()

    low_first        = df["group_num"].isin([1,3,5,7]).sum()
    high_first       = df["group_num"].isin([2,4,6,8]).sum()

    return (
        f"[FLOW - {paradigm.upper()} - GROUP ORDER]\n"
        f"Out of the {n} participants, {low_first} played in low flow first while "
        f"{high_first} played in high flow first.\n"
        f"{rankin_first} played with the Rankin map first while {compressed_first} "
        f"played with the Compressed map first.\n"
        f"Finally, {oddball_a_first} had Oddball A first while {oddball_b_first} "
        f"had Oddball B first."
    )


In [4]:
# ==============================================================================
# Generate all sentences
# ==============================================================================

CONFIGS = [
    ("flow",  "active"),
    ("flow",  "passive"),
    ("load",  "active"),
    ("load",  "passive"),
]

print("=" * 60)
print("DEMOGRAPHICS & VOLUME")
print("=" * 60)
for paper, paradigm in CONFIGS:
    print(demographics_sentence(paper, paradigm))
    print(volume_sentence(paper, paradigm))
    print(game_lvl_sentence(paper, paradigm))
    print()

print("=" * 60)
print("GROUP ORDER (FLOW ONLY)")
print("=" * 60)
for paradigm in ["active", "passive"]:
    print(group_order_sentence(paradigm))
    print()


DEMOGRAPHICS & VOLUME
[FLOW - ACTIVE]
36 participants were included in the analyses (10 female; M = 22.0 years old, SD = 2.4, range 18-28).
Sound volume was individually calibrated and ranged from 56.8 to 65.7 dB SPL (M = 59.3, SD = 2.0).
Game level was individually calibrated, ranging from 2 to 4 (level 2: n = 5; level 3: n = 21; level 4: n = 10).

[FLOW - PASSIVE]
41 participants were included in the analyses (11 female; M = 23.4 years old, SD = 3.7, range 18-33).
Sound volume was individually calibrated (n=40 with volume data; missing: ['S58']) and ranged from 56.7 to 72.3 dB SPL (M = 61.6, SD = 4.4).
Game level was individually calibrated, ranging from 3 to 5 (level 3: n = 5; level 4: n = 27; level 5: n = 9).

[LOAD - ACTIVE]
48 participants were included in the analyses (14 female; M = 22.4 years old, SD = 2.8, range 18-29).
Sound volume was individually calibrated and ranged from 56.8 to 65.7 dB SPL (M = 59.2, SD = 2.0).
Game level was individually calibrated, ranging from 2 to 4